# Day 098 Solution — Trading Bot Launch Pipeline

In [ ]:
import csv, datetime, io, json, pathlib, tempfile
from dataclasses import dataclass, field

@dataclass
class LandingPageConfig:
    product_name: str; tagline: str; description: str; features: list
    cta_text: str = "Join the Waitlist"; cta_url: str = "#waitlist"
    primary_color: str = "#2563eb"

_CFG = LandingPageConfig(
    product_name  = "AI Trading Bot",
    tagline       = "Automate your trading strategy with AI-powered signals.",
    description   = "Built with sentiment analysis, technical indicators, and risk controls.",
    features      = ["Sentiment-driven signals", "Stop-loss protection", "Daily scheduling"],
)
def generate_landing_page(cfg):
    feature_items = "\n".join(f"      <li>{f}</li>" for f in cfg.features)
    return (
        "<!DOCTYPE html>\n"
        '<html lang=\"en\">\n'
        "<head>\n"
        '  <meta charset=\"UTF-8\">\n'
        f"  <title>{cfg.product_name}</title>\n"
        "  <style>\n"
        f"    .hero {{ background: {cfg.primary_color}; color: white; padding: 80px 40px; text-align: center; }}\n"
        "    h1 { font-size: 3rem; margin: 0 0 16px; }\n"
        f"    .cta {{ background: white; color: {cfg.primary_color}; padding: 16px 32px; border-radius: 8px; }}\n"
        "  </style>\n"
        "</head>\n"
        "<body>\n"
        '  <section class=\"hero\">\n'
        f"    <h1>{cfg.product_name}</h1>\n"
        f'    <p class=\"tagline\">{cfg.tagline}</p>\n'
        f'    <a class=\"cta\" href=\"{cfg.cta_url}\">{cfg.cta_text}</a>\n'
        "  </section>\n"
        '  <section class=\"features\">\n'
        "    <h2>Features</h2>\n"
        "    <ul>\n"
        f"{feature_items}\n"
        "    </ul>\n"
        f"    <p>{cfg.description}</p>\n"
        "  </section>\n"
        '  <section id=\"waitlist\">\n'
        "    <h2>Join the Waitlist</h2>\n"
        '    <input type=\"email\" placeholder=\"your@email.com\" />\n'
        f"    <button type=\"button\">{cfg.cta_text}</button>\n"
        "  </section>\n"
        "</body>\n"
        "</html>"
    )
@dataclass
class WaitlistEntry:
    email: str; name: str = ""
    joined_at: str = field(default_factory=lambda: datetime.datetime.now().isoformat())
    source: str = "landing_page"

class Waitlist:
    def __init__(self, path):
        self.path = pathlib.Path(path); self._entries = []
        if self.path.exists(): self._load()
    def _load(self):
        data = json.loads(self.path.read_text(encoding="utf-8"))
        self._entries = [WaitlistEntry(**e) for e in data]
    def _save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        data = [{"email": e.email,"name":e.name,"joined_at":e.joined_at,"source":e.source}
                for e in self._entries]
        self.path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    def add(self, email, name="", source="landing_page"):
        email = email.strip().lower()
        if not email or "@" not in email: raise ValueError(f"Invalid email: {email!r}")
        if any(e.email == email for e in self._entries): return False
        self._entries.append(WaitlistEntry(email=email, name=name, source=source))
        self._save(); return True
    def count(self): return len(self._entries)
    def list_emails(self): return [e.email for e in self._entries]
    def export_csv(self):
        buf = io.StringIO()
        w = csv.DictWriter(buf, fieldnames=["email","name","joined_at","source"])
        w.writeheader()
        for e in self._entries:
            w.writerow({"email":e.email,"name":e.name,"joined_at":e.joined_at,"source":e.source})
        return buf.getvalue()
def generate_tagline_prompt(product_name, description, n=3):
    return [
        {"role":"system","content":(
            "You are a SaaS copywriter. Respond with a JSON array of strings only. "
            "No explanation. No markdown. Just the JSON array."
        )},
        {"role":"user","content":(
            f"Write {n} punchy taglines for a product called '{product_name}'. "
            f"Product description: {description}. "
            f"Each tagline: 10 words or fewer, clear benefit, no jargon. "
            f"Return as a JSON array of {n} strings."
        )},
    ]

def parse_taglines(llm_response):
    text = llm_response.strip()
    if text.startswith("```"):
        lines = text.split("\n"); text = "\n".join(lines[1:-1]) if len(lines) > 2 else text
    return json.loads(text)

def generate_launch_email(product_name, tagline, cta_url, recipient_name=""):
    greeting = f"Hi {recipient_name}," if recipient_name else "Hi there,"
    return {
        "subject": f"{product_name} is live \u2014 you're in!",
        "body": (
            f"{greeting}\n\nThe wait is over. {product_name} is officially live.\n\n"
            f"{tagline}\n\nAs an early member of our waitlist, you get first access.\n\n"
            f"\u2192 {cta_url}\n\nQuestions? Just reply to this email.\n\n"
            f"\u2014 The {product_name} Team"
        ),
    }

def generate_social_post(product_name, tagline, cta_url, platform="twitter"):
    if platform == "twitter":
        post = f"\U0001f680 {product_name} is live!\n\n{tagline}\n\n{cta_url}"
        return post[:277] + "..." if len(post) > 280 else post
    if platform == "linkedin":
        return (
            f"Excited to announce that {product_name} is officially live! \U0001f389\n\n"
            f"{tagline}\n\nWe built this to solve a real problem.\n\n"
            f"Try it here: {cta_url}\n\n#AI #ProductLaunch #BuildInPublic"
        )
    return f"{product_name}: {tagline} \u2014 {cta_url}"


In [ ]:
import os, tempfile

with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
    wl_path = f.name
try: os.unlink(wl_path)
except: pass

PRODUCT = "AI Trading Bot"
TAGLINE = "Automate your strategy with AI-driven signals."
URL     = "https://aitradingbot.io"

html = generate_landing_page(_CFG)
assert html.startswith("<!DOCTYPE html")
assert PRODUCT in html
for feat in _CFG.features:
    assert feat in html

wl = Waitlist(wl_path)
for addr, name in [("alice@test.com","Alice"),("bob@test.com","Bob"),("carol@test.com","Carol")]:
    wl.add(addr, name=name)
assert wl.count() == 3
assert "alice@test.com" in wl.list_emails()

mock_resp = '```json\n["Trade smarter with AI.", "Your edge, automated.", "Signals, not noise."]\n```'
taglines = parse_taglines(mock_resp)
assert len(taglines) == 3

for entry in wl._entries:
    em = generate_launch_email(PRODUCT, TAGLINE, URL, entry.name)
    assert "subject" in em and "body" in em
    assert PRODUCT in em["subject"]

tw = generate_social_post(PRODUCT, TAGLINE, URL, platform="twitter")
li = generate_social_post(PRODUCT, TAGLINE, URL, platform="linkedin")
assert len(tw) <= 280
assert "#AI" in li or "#ProductLaunch" in li

print(f"HTML: {len(html)} chars | Subscribers: {wl.count()} | Taglines: {taglines}")
print(f"Twitter ({len(tw)} chars): {tw}")
print("\nSolution smoke-test passed.")

try: os.unlink(wl_path)
except: pass
